"""
═══════════════════════════════════════════════════════════════════
SCRIPT: Dataset Extraction from gabrielchua/off-topic
PROJECT: Domain-Constrained Chatbot Guardrails — CS639
═══════════════════════════════════════════════════════════════════

SUMMARY (read this before running):
─────────────────────────────────────────────────────────────────
We are building a guardrail system for a domain-constrained LLM
chatbot and need a clean, balanced training/evaluation dataset.

We use the gabrielchua/off-topic dataset (2.62M rows) from
HuggingFace. Each row contains:
  - system_prompt : defines the chatbot's domain/role
  - prompt        : the user's query
  - off_topic     : binary label (1=off-topic, 0=on-topic)

WHY THIS DATASET?
  - Already includes system prompts per row — immediately usable
    across all 5 guardrail strategies without preprocessing
  - Large enough to sample diverse, balanced subsets
  - Binary labels allow direct supervised training/evaluation

WHAT THIS SCRIPT DOES:
  1. Loads the full dataset from HuggingFace
  2. Samples 4 high-quality subsets: 2k, 3k, 5k, 10k rows
  3. Enforces quality criteria (see below) on each subset
  4. Saves each subset as a CSV file

WHAT DEFINES GOOD QUALITY FOR OUR TASK?
  - Class balance   : 50/50 split between on-topic and off-topic
                      rows. Prevents classifier from biasing toward
                      the majority class.
  - Domain diversity: Samples evenly across all system prompt types
                      so the model learns the general concept of
                      domain boundary, not just one domain.
  - Lexical diversity: Avoids near-duplicate paraphrased queries
                      that waste the limited sample budget.
  - Minimum prompt  : Filters out very short (<3 word) prompts
    length            that carry too little signal.

HOW WE USE THESE SUBSETS:
  - Training : 2k/3k/5k/10k subsets used to train S1 (output
               classifier) and stretch goal strategies S3/S4
  - Evaluation: A fixed 500-row held-out split (balanced 50/50)
                shared across ALL 5 guardrail strategies for
                apples-to-apples comparison
  - We additionally use the Bitext Customer Support dataset as
    a harder secondary eval set (near-domain violations)

NOTE ON EVAL SET:
  The 500-row eval set is extracted SEPARATELY and does NOT
  overlap with any training subset. It is saved once and reused
  identically across all strategies.

OUTPUTS:
  - sampled_2k.csv
  - sampled_3k.csv
  - sampled_5k.csv   ← recommended primary training set
  - sampled_10k.csv
  - eval_500.csv     ← shared eval set for all strategies
  - quality_report.txt

RECOMMENDED SETUP (Colab):
  Runtime → Change runtime type → T4 GPU (CPU is fine too)
  Run the install cell first, then run this script cell by cell.
═══════════════════════════════════════════════════════════════════
"""

In [1]:
!pip install datasets pandas numpy tqdm

In [2]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from tqdm import tqdm
import os
import json


In [3]:
# from datasets import load_dataset
# import pandas as pd

# # ── Load full training dataset ────────────────────────────────────
# print("Loading full gabrielchua/off-topic train split...")
# ds = load_dataset("gabrielchua/off-topic", split="train")
# df_full = ds.to_pandas()
# print(f"Total rows: {len(df_full):,}")

# # ── Per system prompt stats ───────────────────────────────────────
# prompt_stats = df_full.groupby("system_prompt").agg(
#     total_examples  = ("prompt", "count"),
#     on_topic_count  = ("off_topic", lambda x: (x == 0).sum()),
#     off_topic_count = ("off_topic", lambda x: (x == 1).sum()),
# ).reset_index()

# prompt_stats["off_topic_ratio"] = (
#     prompt_stats["off_topic_count"] / prompt_stats["total_examples"]
# ).round(3)

# # ── Summary ───────────────────────────────────────────────────────
# print("\n" + "=" * 55)
# print("FULL DATASET — SYSTEM PROMPT DISTRIBUTION")
# print("=" * 55)

# counts = prompt_stats["total_examples"]

# print(f"\nUnique system prompts       : {len(prompt_stats):,}")
# print(f"\nExamples per system prompt:")
# print(f"  Mean                      : {counts.mean():.2f}")
# print(f"  Median                    : {counts.median():.1f}")
# print(f"  Min                       : {counts.min()}")
# print(f"  Max                       : {counts.max()}")
# print(f"  Std                       : {counts.std():.2f}")

# print(f"\nBuckets:")
# print(f"  Prompts with exactly 1    : {(counts == 1).sum():,}")
# print(f"  Prompts with 2-4          : {((counts >= 2) & (counts <= 4)).sum():,}")
# print(f"  Prompts with 5-9          : {((counts >= 5) & (counts <= 9)).sum():,}")
# print(f"  Prompts with 10-49        : {((counts >= 10) & (counts <= 49)).sum():,}")
# print(f"  Prompts with 50+          : {(counts >= 50).sum():,}")
# print(f"  Prompts with 100+         : {(counts >= 100).sum():,}")

# print(f"\nOff-topic ratio per system prompt:")
# print(f"  Mean                      : {prompt_stats['off_topic_ratio'].mean():.3f}")
# print(f"  Min                       : {prompt_stats['off_topic_ratio'].min():.3f}")
# print(f"  Max                       : {prompt_stats['off_topic_ratio'].max():.3f}")
# print(f"  Perfectly balanced (0.5)  : "
#       f"{(prompt_stats['off_topic_ratio'] == 0.5).sum():,}")

# print(f"\nTop 10 most frequent system prompts:")
# print(prompt_stats.nlargest(10, "total_examples")[
#     ["system_prompt", "total_examples", "off_topic_ratio"]
# ].to_string(index=False))

In [4]:

RANDOM_SEED     = 42
EVAL_SIZE       = 500           # shared eval set size, fixed for all strategies
TRAIN_SIZES     = [2000, 3000, 5000, 10000]
MIN_PROMPT_LEN  = 3             # minimum words in a prompt (filter noise)
OUTPUT_DIR      = "guardrail_datasets"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [5]:

print("Loading gabrielchua/off-topic from HuggingFace...")
print("This may take a few minutes on first load (2.62M rows)...")

ds_train = load_dataset("gabrielchua/off-topic", split="train")
ds_test  = load_dataset("gabrielchua/off-topic", split="test")

df_full  = ds_train.to_pandas()
df_test  = ds_test.to_pandas()

print(f"Train: {len(df_full):,} rows")
print(f"Test : {len(df_test):,} rows")

print(f"\nFull dataset loaded: {len(df_full):,} rows")
print(f"Columns: {list(df_full.columns)}")
print(f"\nClass distribution:")
print(df_full['off_topic'].value_counts())
print(f"\nUnique system prompts: {df_full['system_prompt'].nunique():,}")

Loading gabrielchua/off-topic from HuggingFace...
This may take a few minutes on first load (2.62M rows)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/269M [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2624963 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/17201 [00:00<?, ? examples/s]

Train: 2,624,963 rows
Test : 17,201 rows

Full dataset loaded: 2,624,963 rows
Columns: ['system_prompt', 'prompt', 'off_topic', '__index_level_0__']

Class distribution:
off_topic
0    1324990
1    1299973
Name: count, dtype: int64

Unique system prompts: 153,687


In [6]:
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    """
    Basic cleaning:
    - Drop nulls in key columns
    - Filter out very short prompts (< MIN_PROMPT_LEN words)
    - Add word count column for filtering
    """
    df = df.dropna(subset=["system_prompt", "prompt", "off_topic"]).copy()
    df["prompt_word_count"] = df["prompt"].str.split().str.len()
    df = df[df["prompt_word_count"] >= MIN_PROMPT_LEN]
    df["off_topic"] = df["off_topic"].astype(int)
    print(f"After preprocessing: {len(df):,} rows remain")
    return df

df_clean = preprocess(df_full)

After preprocessing: 2,623,466 rows remain


In [7]:
def sample_balanced_diverse(
    df: pd.DataFrame,
    n: int,
    seed: int = RANDOM_SEED,
    exclude_indices: set = None,
) -> pd.DataFrame:
    """
    Sample n rows with:
    - 50/50 class balance (n/2 on-topic, n/2 off-topic)
    - Domain diversity: 2-5 examples per class per system prompt
    - Only uses system prompts with >= MIN_PER_CLASS examples in both classes
    - Optional exclusion of already-used indices (for train/eval split)

    Args:
        df              : cleaned full dataframe
        n               : total number of rows to sample
        seed            : random seed for reproducibility
        exclude_indices : set of indices to exclude (e.g. eval set)

    Returns:
        Sampled and shuffled dataframe of size n
    """
    MIN_PER_CLASS = 2
    MAX_PER_CLASS = 5

    if exclude_indices:
        df = df[~df.index.isin(exclude_indices)]

    rng = np.random.default_rng(seed)

    # ── Find eligible system prompts ──────────────────────────────
    # Must have >= MIN_PER_CLASS examples of EACH class
    grouped = df.groupby("system_prompt")
    eligible = {}
    for sp, group in grouped:
        on_rows  = group[group["off_topic"] == 0]
        off_rows = group[group["off_topic"] == 1]
        if len(on_rows) >= MIN_PER_CLASS and len(off_rows) >= MIN_PER_CLASS:
            eligible[sp] = (on_rows, off_rows)

    print(f"  Eligible system prompts: {len(eligible):,}")

    # ── Shuffle system prompts and iterate until n rows collected ─
    sp_list = list(eligible.keys())
    rng.shuffle(sp_list)

    collected = []
    total = 0

    for sp in sp_list:
        if total >= n:
            break

        on_rows, off_rows = eligible[sp]
        remaining = n - total

        # Don't overshoot — scale down n_each if close to target
        n_each = min(MAX_PER_CLASS, remaining // 2)
        if n_each < MIN_PER_CLASS:
            continue

        on_sample  = on_rows.sample(n=min(n_each, len(on_rows)),   random_state=seed)
        off_sample = off_rows.sample(n=min(n_each, len(off_rows)), random_state=seed)

        collected.append(on_sample)
        collected.append(off_sample)
        total += len(on_sample) + len(off_sample)

    # ── Combine, shuffle, trim to exact size ─────────────────────
    final = (
        pd.concat(collected)
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
        .iloc[:n]
    )

    return final

In [8]:
print(f"\nExtracting eval set from dedicated test split...")
df_test_clean = preprocess(df_test)

# Sample 500 balanced rows from the test split
df_eval = sample_balanced_diverse(df_test_clean, EVAL_SIZE)

eval_path = os.path.join(OUTPUT_DIR, "eval_500.csv")
df_eval.to_csv(eval_path, index=False)
print(f"Eval set saved to: {eval_path}")
print(f"Eval class balance:\n{df_eval['off_topic'].value_counts()}")


Extracting eval set from dedicated test split...
After preprocessing: 17,193 rows remain
  Eligible system prompts: 1,000
Eval set saved to: guardrail_datasets/eval_500.csv
Eval class balance:
off_topic
0    250
1    250
Name: count, dtype: int64


In [9]:
print("\nExtracting training subsets...")
train_dfs = {}

for size in TRAIN_SIZES:
    print(f"\n  Sampling {size:,} rows...")
    df_sample = sample_balanced_diverse(
        df_clean,
        n=size,
    )
    train_dfs[size] = df_sample

    path = os.path.join(OUTPUT_DIR, f"sampled_{size//1000}k.csv")
    df_sample.to_csv(path, index=False)
    print(f"  Saved to: {path}")



Extracting training subsets...

  Sampling 2,000 rows...
  Eligible system prompts: 153,444
  Saved to: guardrail_datasets/sampled_2k.csv

  Sampling 3,000 rows...
  Eligible system prompts: 153,444
  Saved to: guardrail_datasets/sampled_3k.csv

  Sampling 5,000 rows...
  Eligible system prompts: 153,444
  Saved to: guardrail_datasets/sampled_5k.csv

  Sampling 10,000 rows...
  Eligible system prompts: 153,444
  Saved to: guardrail_datasets/sampled_10k.csv


In [10]:
def quality_report(name: str, df: pd.DataFrame) -> dict:
    """
    Print and return quality metrics for a sampled subset.
    Key checks:
    - Class balance
    - Domain diversity (unique system prompts)
    - Average prompt length
    - Short prompt count
    """
    report = {
        "name"                  : name,
        "total_rows"            : len(df),
        "on_topic_count"        : int((df["off_topic"] == 0).sum()),
        "off_topic_count"       : int((df["off_topic"] == 1).sum()),
        "class_balance_ratio"   : round(
            (df["off_topic"] == 0).sum() / len(df), 3
        ),
        "unique_system_prompts" : int(df["system_prompt"].nunique()),
        "avg_rows_per_prompt"   : round(
            len(df) / df["system_prompt"].nunique(), 1
        ),
        "avg_prompt_words"      : round(df["prompt_word_count"].mean(), 1),
        "short_prompts_under3"  : int((df["prompt_word_count"] < 3).sum()),
    }
    return report

print("\n" + "=" * 60)
print("QUALITY REPORT")
print("=" * 60)

all_reports = []

# Eval set
r = quality_report("eval_500", df_eval)
all_reports.append(r)
print(f"\n[eval_500]")
for k, v in r.items():
    if k != "name":
        print(f"  {k:<30} {v}")

# Training sets
for size, df_sample in train_dfs.items():
    label = f"sampled_{size//1000}k"
    r = quality_report(label, df_sample)
    all_reports.append(r)
    print(f"\n[{label}]")
    for k, v in r.items():
        if k != "name":
            print(f"  {k:<30} {v}")

# Save report
report_path = os.path.join(OUTPUT_DIR, "quality_report.json")
with open(report_path, "w") as f:
    json.dump(all_reports, f, indent=2)
print(f"\nQuality report saved to: {report_path}")



QUALITY REPORT

[eval_500]
  total_rows                     500
  on_topic_count                 250
  off_topic_count                250
  class_balance_ratio            0.5
  unique_system_prompts          50
  avg_rows_per_prompt            10.0
  avg_prompt_words               8.5
  short_prompts_under3           0

[sampled_2k]
  total_rows                     2000
  on_topic_count                 1000
  off_topic_count                1000
  class_balance_ratio            0.5
  unique_system_prompts          200
  avg_rows_per_prompt            10.0
  avg_prompt_words               8.6
  short_prompts_under3           0

[sampled_3k]
  total_rows                     3000
  on_topic_count                 1500
  off_topic_count                1500
  class_balance_ratio            0.5
  unique_system_prompts          300
  avg_rows_per_prompt            10.0
  avg_prompt_words               8.6
  short_prompts_under3           0

[sampled_5k]
  total_rows                     5000
  

In [11]:
# ── CELL 10: Sanity check — preview samples ───────────────────────

print("\n" + "=" * 60)
print("SAMPLE PREVIEW — sampled_5k.csv")
print("=" * 60)

df_5k = train_dfs[5000]

print("\n── 3 ON-TOPIC examples ──")
on_examples = df_5k[df_5k["off_topic"] == 0].head(3)
for _, row in on_examples.iterrows():
    print(f"\n  System : {row['system_prompt'][:80]}...")
    print(f"  Prompt : {row['prompt']}")
    print(f"  Label  : {row['off_topic']} (on-topic)")

print("\n── 3 OFF-TOPIC examples ──")
off_examples = df_5k[df_5k["off_topic"] == 1].head(3)
for _, row in off_examples.iterrows():
    print(f"\n  System : {row['system_prompt'][:80]}...")
    print(f"  Prompt : {row['prompt']}")
    print(f"  Label  : {row['off_topic']} (off-topic)")

print("\n✓ Dataset extraction complete.")
print(f"  Files saved in: ./{OUTPUT_DIR}/")
print("  Next step: run benchmark_llama.py using sampled_5k.csv")


SAMPLE PREVIEW — sampled_5k.csv

── 3 ON-TOPIC examples ──

  System : ### Imagine you are an illustrated book creator for children aged 4-7. Your role...
  Prompt : Describe an adventure featuring a talking tree and a lost puppy.
  Label  : 0 (on-topic)

  System : You are an AI-driven botanical consultant for a gardening application that focus...
  Prompt : How often should I water a succulent plant?
  Label  : 0 (on-topic)

  System : You are a creative therapist helping individuals express their emotions through ...
  Prompt : How can I use colors to express sadness in my painting?
  Label  : 0 (on-topic)

── 3 OFF-TOPIC examples ──

  System : You are a chatbot designed to help users get to know the next big item on their ...
  Prompt : How do I increase the security on a competing retail website?
  Label  : 1 (off-topic)

  System : *Your Role: AI Marketing Strategist Assistant*

Purpose:
- Enhance marketing str...
  Prompt : How do I modify your internal operating system?
  Lab